### TFT Freight Cost Prediction — Any Route, Any Year

Loads the saved model bundle and `df_model.parquet` from disk.  
Specify any **origin, destination, product code, and forecast year**.  
- If the route was seen during training → predicts directly.  
- If the route is **unseen** → selects the best proxy route (same product + origin if possible) and uses the actual covariate features for the requested route (or product-year averages if the route has no data at all).


In [30]:
ORIGIN        = "Japan"           # exact Origin_Label string
DESTINATION   = "China"   # exact Destination_Label string
PRODUCT       = 8517              # Product_Code integer (e.g. 8517, 2106, 3304, 9404, 6109)
FORECAST_YEAR = 2022              # any year; years beyond data → out-of-sample forecast

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  Edit these four lines to predict any route / year
# ══════════════════════════════════════════════════════════════════════════════


# ── Paths (relative to notebook; adjust if your layout differs) ───────────────
SAVE_DIR      = "saved_models"
BUNDLE_PATH   = os.path.join(SAVE_DIR, "tft_model2_all_products_bundle.pt")
DF_MODEL_PATH = os.path.join(SAVE_DIR, "df_model.parquet")
# Optional: if you saved df_all separately (recommended), put the path here:
DF_ALL_PATH   = os.path.join(SAVE_DIR, "df_all_tft_filtered.parquet")

# ── TFT column names (must match what the model was trained with) ─────────────
TIME_COL = "Year"
TARGET   = "y_log"

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Load bundle & model from disk
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 62)
print("  Loading saved bundle ...")
bundle          = torch.load(BUNDLE_PATH, weights_only=False)
tft_inf         = TemporalFusionTransformer.load_from_checkpoint(bundle["model_ckpt"])
tft_inf.eval()

training_ds_inf = bundle["training_ds"]
min_yr_inf      = int(bundle["min_year"])

print(f"  Checkpoint : {bundle['model_ckpt']}")
print(f"  Products   : {bundle['products']}")
print(f"  Min year   : {min_yr_inf}")


  Loading saved bundle ...


c:\Users\xianj\anaconda3\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\xianj\anaconda3\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


  Checkpoint : saved_models\tft_model2_all_products.ckpt
  Products   : [np.int64(2106), np.int64(3304), np.int64(6109), np.int64(8517), np.int64(9404)]
  Min year   : 2016


In [31]:
from sklearn.preprocessing import StandardScaler

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — Load df_model from parquet  (ground-truth covariate source)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n  Loading data from {DF_MODEL_PATH} ...")
df_raw = pd.read_parquet(DF_MODEL_PATH).copy()
df_raw[TIME_COL] = df_raw[TIME_COL].astype(int)
df_raw["group_id"] = (
    df_raw["Origin_Label"].astype(str) + "||" +
    df_raw["Destination_Label"].astype(str) + "||" +
    df_raw["Product_Code"].astype(str)
).astype(str)

# Infer covariate columns from training dataset schema
trained_tvur   = list(training_ds_inf.time_varying_unknown_reals)
COVARIATE_COLS = [c for c in trained_tvur if c != TARGET]
print(f"  Covariate columns used by model : {len(COVARIATE_COLS)}")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — Determine known routes (groups the model can score directly)
# ══════════════════════════════════════════════════════════════════════════════
def _known_groups_from_encoder(ds):
    for attr in ("categorical_encoders", "_categorical_encoders", "scalers"):
        mapping = getattr(ds, attr, None)
        if not isinstance(mapping, dict):
            continue
        enc = mapping.get("group_id")
        if enc is None:
            continue
        classes = getattr(enc, "classes_", None)
        if classes is None:
            continue
        return set(classes.keys()) if hasattr(classes, "keys") else set(classes)
    return None


known_groups = _known_groups_from_encoder(training_ds_inf)

if known_groups:
    print(f"  Known routes (from encoder)    : {len(known_groups)}")
else:
    if os.path.exists(DF_ALL_PATH):
        df_all_disk = pd.read_parquet(DF_ALL_PATH).copy()
        if "group_id" not in df_all_disk.columns:
            df_all_disk["group_id"] = (
                df_all_disk["Origin_Label"].astype(str) + "||" +
                df_all_disk["Destination_Label"].astype(str) + "||" +
                df_all_disk["Product_Code"].astype(str)
            ).astype(str)
        known_groups = set(df_all_disk["group_id"].unique())
        print(f"  Known routes (from df_all parquet) : {len(known_groups)}")
    else:
        min_len  = training_ds_inf.max_encoder_length + training_ds_inf.max_prediction_length + 1
        max_yr   = int(df_raw[TIME_COL].max())
        split_yr = max_yr - 2
        grp_stats    = df_raw.groupby("group_id")[TIME_COL].agg(["count", "max"])
        mask         = (grp_stats["count"] >= min_len) & (grp_stats["max"] > split_yr)
        known_groups = set(grp_stats[mask].index.tolist())
        print(
            f"  ⚠  Encoder not readable. Approximating known groups from parquet "
            f"(≥{min_len} years, post-{split_yr} data): {len(known_groups)}\n"
            f"     For exact results, save df_all to '{DF_ALL_PATH}' from the training notebook."
        )

if not known_groups:
    raise RuntimeError(
        "Could not determine any known groups. "
        f"Save df_all to disk with:\n  df_all.to_parquet('{DF_ALL_PATH}')\nand re-run this cell."
    )

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — Similarity-based proxy selection helpers
# ══════════════════════════════════════════════════════════════════════════════
SIMILARITY_COLS = [
    "dist_km",
    "dest_teu",
    "lsci_d",
    "Destination_GDP_US_at_current_prices_in_millions_Value",
    "trade_imbalance",
]

def _build_target_profile(origin, destination, product, df):
    """
    Build a representative logistics profile for any (origin, destination,
    product) triple, even if it has no rows in df.

    Strategy (in order of priority):
      1. Use actual rows for the exact route (best case).
      2. Impute from destination-level averages + origin-level dist_km.
      3. Fill any remaining NaNs with the global column mean.
    """
    exact = df[df["group_id"] == f"{origin}||{destination}||{str(product)}"]
    if not exact.empty:
        profile = exact[SIMILARITY_COLS].mean()
        if not profile.isna().all():
            return profile

    # Fallback: piece together from destination / origin averages
    dest_rows   = df[df["Destination_Label"] == destination]
    origin_rows = df[df["Origin_Label"] == origin]

    profile = {}

    # Destination-level features
    for col in ["dest_teu", "lsci_d",
                "Destination_GDP_US_at_current_prices_in_millions_Value",
                "trade_imbalance"]:
        profile[col] = dest_rows[col].mean() if not dest_rows.empty else np.nan

    # dist_km: try same O-D pair across all products first, then origin average
    od_rows = df[(df["Origin_Label"] == origin) & (df["Destination_Label"] == destination)]
    if not od_rows.empty:
        profile["dist_km"] = od_rows["dist_km"].mean()
    elif not origin_rows.empty:
        profile["dist_km"] = origin_rows["dist_km"].mean()
    else:
        profile["dist_km"] = df["dist_km"].mean()   # global average last resort

    series = pd.Series({c: profile.get(c, np.nan) for c in SIMILARITY_COLS})

    # Fill any surviving NaNs with global column means
    for col in SIMILARITY_COLS:
        if pd.isna(series[col]):
            series[col] = df[col].mean()

    return series


def find_best_logistics_proxy(target_profile, candidate_pool_rows):
    """
    Return the group_id from candidate_pool_rows whose mean logistics profile
    is closest to target_profile (normalised Euclidean distance).

    Parameters
    ----------
    target_profile      : pd.Series indexed by SIMILARITY_COLS
    candidate_pool_rows : DataFrame containing known-route rows
    """
    candidate_profiles = (
        candidate_pool_rows
        .groupby("group_id")[SIMILARITY_COLS]
        .mean()
        .dropna(how="all")
    )

    if candidate_profiles.empty:
        return sorted(candidate_pool_rows["group_id"].unique())[0]

    # Stack target + candidates, fill any NaNs with column means, then scale
    combined = pd.concat(
        [target_profile.to_frame().T, candidate_profiles],
        ignore_index=False
    )
    combined = combined.fillna(combined.mean())

    scaler            = StandardScaler()
    scaled            = scaler.fit_transform(combined)
    scaled_target     = scaled[0]
    scaled_candidates = scaled[1:]

    distances = np.linalg.norm(scaled_candidates - scaled_target, axis=1)
    return candidate_profiles.index[int(np.argmin(distances))]


# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 — Identify the requested route; find proxy if unseen
# ══════════════════════════════════════════════════════════════════════════════
REQ_GROUP_ID = f"{ORIGIN}||{DESTINATION}||{PRODUCT}"
using_proxy  = False
proxy_note   = ""

req_rows = df_raw[df_raw["group_id"] == REQ_GROUP_ID].copy()

if REQ_GROUP_ID in known_groups:
    active_group = REQ_GROUP_ID
    print(f"\n  Route '{REQ_GROUP_ID}' is KNOWN to the model — predicting directly.")
else:
    print(f"\n  Route '{REQ_GROUP_ID}' is NOT in the model. Finding best proxy ...")

    # Build target profile even when route has no rows in parquet
    target_profile = _build_target_profile(ORIGIN, DESTINATION, PRODUCT, df_raw)
    print(f"  Target profile   : {target_profile.round(2).to_dict()}")

    # Determine product category of the requested route
    if not req_rows.empty and "product_cat" in req_rows.columns:
        req_product_cat = req_rows["product_cat"].iloc[0]
    else:
        cat_rows = df_raw[df_raw["Product_Code"] == PRODUCT]
        req_product_cat = cat_rows["product_cat"].iloc[0] if not cat_rows.empty else None

    pool_mask = df_raw["group_id"].isin(known_groups)

    if req_product_cat is not None and "product_cat" in df_raw.columns:
        cat_mask  = df_raw["product_cat"] == req_product_cat
        prod_mask = df_raw["Product_Code"] == PRODUCT

        for label, extra_mask in [
            ("same product",          pool_mask & prod_mask),
            ("same product category", pool_mask & cat_mask),
            ("all known routes",      pool_mask),
        ]:
            candidate_pool = df_raw[extra_mask]
            if candidate_pool["group_id"].nunique() > 0:
                scope_label = label
                break
    else:
        candidate_pool = df_raw[pool_mask]
        scope_label    = "all known routes"

    active_group = find_best_logistics_proxy(target_profile, candidate_pool)

    proxy_note = f"{scope_label} · best logistics match: {active_group}"
    using_proxy = True
    print(f"  Candidate pool   : {scope_label}  "
          f"({candidate_pool['group_id'].nunique()} unique routes)")
    print(f"  Best proxy       : {active_group}")
    print(f"  ⚠  Predictions use the proxy's learned route embedding, "
          f"not '{ORIGIN}→{DESTINATION}'.")

proxy_origin, proxy_dest, proxy_prod = active_group.split("||")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6 — Build encoder history rows
# ══════════════════════════════════════════════════════════════════════════════
context_end = FORECAST_YEAR - 1
proxy_rows  = df_raw[df_raw["group_id"] == active_group].copy()

if not req_rows.empty:
    context_src    = req_rows
    covariate_note = "requested route (from parquet)"
else:
    context_src    = proxy_rows
    covariate_note = f"proxy route '{active_group}'"

context_rows = context_src[context_src[TIME_COL] <= context_end].copy()

if context_rows.empty:
    raise ValueError(
        f"No historical data up to {context_end} for '{REQ_GROUP_ID}' "
        f"or proxy '{active_group}'. Cannot build encoder context."
    )

# Relabel to active (proxy) group_id so the model's encoder accepts it
context_rows["group_id"]           = active_group
context_rows["Origin_Label"]       = proxy_origin
context_rows["Destination_Label"]  = proxy_dest
context_rows["Product_Code"]       = int(proxy_prod)
context_rows["time_idx"]           = (context_rows[TIME_COL] - min_yr_inf).astype(int)
context_rows = context_rows.sort_values("time_idx").reset_index(drop=True)

print(f"\n  Encoder context  : years {sorted(context_rows[TIME_COL].tolist())}")
print(f"  Covariate source : {covariate_note}")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 7 — Build the forecast row for FORECAST_YEAR
# ══════════════════════════════════════════════════════════════════════════════
actual_row   = req_rows[req_rows[TIME_COL] == FORECAST_YEAR]
has_actual   = not actual_row.empty
ground_truth = float(actual_row[TARGET].iloc[0]) if has_actual else None

last_ctx = context_rows.iloc[[-1]].copy()
fcst_row = last_ctx.copy()
fcst_row[TIME_COL]   = FORECAST_YEAR
fcst_row["time_idx"] = int(FORECAST_YEAR - min_yr_inf)

if has_actual:
    for col in COVARIATE_COLS:
        if col in actual_row.columns:
            fcst_row[col] = actual_row[col].iloc[0]
    fcst_row[TARGET] = last_ctx[TARGET].iloc[0]
else:
    prod_yr_rows = df_raw[
        (df_raw["Product_Code"].astype(str) == proxy_prod) &
        (df_raw[TIME_COL] == FORECAST_YEAR)
    ]
    if not prod_yr_rows.empty:
        prod_mean = prod_yr_rows[COVARIATE_COLS].mean()
        for col in COVARIATE_COLS:
            if col in prod_mean.index and not pd.isna(prod_mean[col]):
                fcst_row[col] = prod_mean[col]
    fcst_row[TARGET] = last_ctx[TARGET].iloc[0]

fcst_row["group_id"]           = active_group
fcst_row["Origin_Label"]       = proxy_origin
fcst_row["Destination_Label"]  = proxy_dest
fcst_row["Product_Code"]       = int(proxy_prod)

df_demo = (
    pd.concat([context_rows, fcst_row], ignore_index=True)
    .sort_values("time_idx")
    .reset_index(drop=True)
)

print(f"  Inference rows   : {len(df_demo)}  "
      f"(years {df_demo[TIME_COL].min()}–{df_demo[TIME_COL].max()})")
print(f"  Ground truth for {FORECAST_YEAR}: "
      f"{f'{ground_truth:.4f}' if has_actual else 'not available (out-of-sample)'}")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 8 — Build inference TimeSeriesDataSet & predict
# ══════════════════════════════════════════════════════════════════════════════
demo_ds = TimeSeriesDataSet.from_dataset(
    training_ds_inf, df_demo,
    predict=True, stop_randomization=True, allow_missing_timesteps=True,
)
demo_loader = demo_ds.to_dataloader(train=False, batch_size=64, num_workers=0)

pred_median = tft_inf.predict(
    demo_loader,
    mode="prediction",
    trainer_kwargs={"accelerator": "auto", "logger": False, "enable_progress_bar": False},
)
median_log  = float(pred_median[-1, -1])
median_orig = float(np.expm1(median_log))

pred_q = tft_inf.predict(
    demo_loader,
    mode="quantiles",
    trainer_kwargs={"accelerator": "auto", "logger": False, "enable_progress_bar": False},
)
QUANTILE_NAMES = ["q10", "q20", "q30", "q50", "q70", "q80", "q90"]
q_log_vals = pred_q[-1, -1, :].tolist()

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 9 — Display results
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 62}")
print(f"  TFT Forecast  |  {ORIGIN}  →  {DESTINATION}")
print(f"  Product : {PRODUCT}   |   Forecast Year : {FORECAST_YEAR}")
if using_proxy:
    print(f"  ⚠  Route unseen — proxy ({proxy_note})")
print(f"{'=' * 62}")
print(f"  {'Quantile':<10} {'Log-scale':>12} {'Original scale':>16}")
print(f"  {'-' * 42}")
for qname, qval in zip(QUANTILE_NAMES, q_log_vals):
    tag = "  ◄ median" if qname == "q50" else ""
    print(f"  {qname:<10} {qval:>12.4f} {np.expm1(qval):>16.2f}{tag}")

if has_actual:
    err = median_log - ground_truth
    print(f"\n  Actual y_log    : {ground_truth:.4f}  ({np.expm1(ground_truth):.2f} original scale)")
    print(f"  Predicted (q50) : {median_log:.4f}  ({median_orig:.2f} original scale)")
    print(f"  Error (log)     : {err:+.4f}")
else:
    print(f"\n  (No ground truth for {FORECAST_YEAR} — genuine out-of-sample forecast)")



  Loading data from saved_models\df_model.parquet ...
  Covariate columns used by model : 38
  Known routes (from encoder)    : 25020

  Route 'Japan||China||8517' is KNOWN to the model — predicting directly.

  Encoder context  : years [2016, 2017, 2018, 2019, 2020, 2021]
  Covariate source : requested route (from parquet)
  Inference rows   : 7  (years 2016–2022)
  Ground truth for 2022: not available (out-of-sample)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\xianj\anaconda3\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\xianj\anaconda3\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to i


  TFT Forecast  |  Japan  →  China
  Product : 8517   |   Forecast Year : 2022
  Quantile      Log-scale   Original scale
  ------------------------------------------
  q10              0.0854             0.09
  q20              0.0966             0.10
  q30              0.1322             0.14
  q50              0.1378             0.15  ◄ median
  q70              0.2130             0.24
  q80              0.2207             0.25
  q90              0.2589             0.30

  (No ground truth for 2022 — genuine out-of-sample forecast)


In [16]:
df_raw["Origin_Label"].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra',
       'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina',
       'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan',
       'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus',
       'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan',
       'Bolivia (Plurinational State of)',
       'Bonaire, Sint Eustatius and Saba', 'Bosnia and Herzegovina',
       'Botswana', 'Brazil', 'British Virgin Islands',
       'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi',
       'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Cayman Islands',
       'Central African Republic', 'Chad', 'Chile', 'China',
       'China, Hong Kong SAR', 'China, Macao SAR',
       'China, Taiwan Province of', 'Christmas Island', 'Colombia',
       'Comoros', 'Congo, Dem. Rep. of the', 'Cook Islands', 'Costa Rica',
       "Cote d'Ivoire", 'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia',
       'Denmark', 'Djibouti', 'Domin